In [1]:
import sqlite3
from datetime import date, datetime

conn = sqlite3.connect("attendance.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    student_id INTEGER PRIMARY KEY,
    full_name TEXT NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS attendance (
    attendance_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER NOT NULL,
    attendance_date TEXT NOT NULL,
    status TEXT NOT NULL CHECK(status IN ('Present', 'Absent')),
    UNIQUE(student_id, attendance_date),
    FOREIGN KEY(student_id) REFERENCES students(student_id)
)
""")

students = [
    (1, "Anna Reyes"),
    (2, "Carlos Dela Cruz"),
    (3, "Mika Santos"),
    (4, "John Garcia"),
    (5, "Sarah Lopez"),
    (6, "Peter Cruz"),
    (7, "Angela Torres"),
    (8, "Kevin Tan"),
    (9, "Daniel Flores"),
    (10, "Maria Mendoza")
]

cursor.executemany("""
INSERT OR IGNORE INTO students (student_id, full_name)
VALUES (?, ?)
""", students)

conn.commit()

while True:
    print("\nATTENDANCE RECORDING SYSTEM")
    print("1. Record Attendance")
    print("2. View Attendance by Student")
    print("3. View Attendance by Date")
    print("4. Display All Attendance")
    print("5. Display All Students")
    print("6. Exit")

    choice = input("Choose an option: ")

    if choice == "1":
        try:
            student_id = int(input("Enter Student ID: "))
            attendance_date = input("Enter Date (YYYY-MM-DD): ")
            status = input("Enter Status (Present/Absent): ").capitalize()

            datetime.strptime(attendance_date, "%Y-%m-%d")

            cursor.execute(
                "SELECT full_name FROM students WHERE student_id = ?",
                (student_id,)
            )

            student = cursor.fetchone()

            if not student:
                print("Student not found.")
                continue

            if status not in ["Present", "Absent"]:
                print("Invalid attendance status.")
                continue

            cursor.execute("""
            INSERT INTO attendance
            (student_id, attendance_date, status)
            VALUES (?, ?, ?)
            """, (student_id, attendance_date, status))

            conn.commit()
            print("Attendance successfully recorded.")

        except ValueError:
            print("Invalid input. Please check the ID and date format.")

        except sqlite3.IntegrityError:
            print("Warning: Attendance for this student and date already exists.")

    elif choice == "2":
        student_id = input("Enter Student ID: ")

        cursor.execute("""
        SELECT students.full_name, attendance.attendance_date,
               attendance.status
        FROM attendance
        JOIN students
        ON attendance.student_id = students.student_id
        WHERE students.student_id = ?
        ORDER BY DATE(attendance.attendance_date)
        """, (student_id,))

        results = cursor.fetchall()

        if results:
            for row in results:
                print(row)
        else:
            print("No attendance records found.")

    elif choice == "3":
        attendance_date = input("Enter Date (YYYY-MM-DD): ")

        try:
            datetime.strptime(attendance_date, "%Y-%m-%d")

            cursor.execute("""
            SELECT students.student_id, students.full_name,
                   attendance.status
            FROM attendance
            JOIN students
            ON attendance.student_id = students.student_id
            WHERE DATE(attendance.attendance_date) = DATE(?)
            """, (attendance_date,))

            results = cursor.fetchall()

            if results:
                for row in results:
                    print(row)
            else:
                print("No attendance records found for this date.")

        except ValueError:
            print("Invalid date format.")

    elif choice == "4":
        cursor.execute("""
        SELECT students.student_id, students.full_name,
               attendance.attendance_date, attendance.status
        FROM attendance
        JOIN students
        ON attendance.student_id = students.student_id
        ORDER BY DATE(attendance.attendance_date)
        """)

        for row in cursor.fetchall():
            print(row)

    elif choice == "5":
        cursor.execute("SELECT * FROM students")

        for row in cursor.fetchall():
            print(row)

    elif choice == "6":
        print("Attendance Recording System closed.")
        break

    else:
        print("Invalid choice. Please try again.")

conn.close()


ATTENDANCE RECORDING SYSTEM
1. Record Attendance
2. View Attendance by Student
3. View Attendance by Date
4. Display All Attendance
5. Display All Students
6. Exit
Attendance successfully recorded.

ATTENDANCE RECORDING SYSTEM
1. Record Attendance
2. View Attendance by Student
3. View Attendance by Date
4. Display All Attendance
5. Display All Students
6. Exit
Attendance Recording System closed.
